In [2]:
import pandas as pd

df = pd.read_parquet("hf://datasets/Bingsu/ko_alpaca_data/data/train-00000-of-00001-0ccadda3390e7796.parquet", engine="fastparquet")

df.head(5)

/home/pyc/anaconda3/envs/python312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,instruction,input,output
0,건강을 유지하기 위한 세 가지 팁을 알려주세요.,,"세 가지 팁은 아침식사를 꼭 챙기며, 충분한 수면을 취하고, 적극적으로 운동을 하는..."
1,세 가지 기본 색은 무엇인가요?,,"기본 색은 빨강, 파랑, 노랑입니다."
2,원자의 구조를 설명하세요.,,"원자는 양성자, 중성자, 전자로 구성되어 있으며, 양성자와 중성자는 원자핵 안에 있..."
3,대기 오염을 어떻게 줄일 수 있나요?,,"대기 오염을 줄이기 위해 자동차 대신 대중교통을 이용하고, 미세먼지 배출량을 줄이기..."
4,어려운 결정을 내려야 했던 때를 설명하세요.,,제가 어려운 결정을 내려야 했던 때는 대학원 졸업 후 직장을 찾아야 했던 때입니다....


In [3]:
# Alpaca 형식 판다스 데이터 셋에서 Alpaca 형식 프롬프트를 생성하는 함수를 정의한다.
def format_input(df):
    instruction_text = (
        f"Below is an instruction that describes a task."
        f"Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{df['instruction']}\n\n"
    )
    if df['input']:
        instruction_text += f"### Input:\n{df['input']}\n\n"
    instruction_text += f"### Response:\n{df['output']}"
    return instruction_text

In [4]:
# Alpaca 형식 판다스 데이터 셋에서 Alpaca 형식 프롬프트를 생성하는 함수를 정의한다.
#   - response를 포함하지 않고 instruction과 input만 포함한다.
def format_input_not_contain_response(df):
    instruction_text = (
        f"Below is an instruction that describes a task."
        f"Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{df['instruction']}"
    )
    if df['input']:
        instruction_text += f"\n\n### Input:\n{df['input']}"
    return instruction_text

In [5]:
print(format_input(df.loc[4615]))

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
다음 단어들을 사용하여 문장을 구성하십시오.

### Input:
아름다운 아침

### Response:
아름다운 아침에는 새벽 조깅을 해보세요.


In [6]:
print(format_input(df.loc[0]))

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
건강을 유지하기 위한 세 가지 팁을 알려주세요.

### Response:
세 가지 팁은 아침식사를 꼭 챙기며, 충분한 수면을 취하고, 적극적으로 운동을 하는 것입니다.


In [7]:
# 데이터 프레임을 훈련, 테스트, 검증 데이터 셋으로 0.85, 0.10, 0.05 비율로 나눈다.
print(df.shape)
train_index = int(df.shape[0] * 0.85)
test_index = int(df.shape[0] * 0.10)
val_index = df.shape[0] - train_index - test_index

print(train_index, test_index, val_index)
print(train_index + test_index + val_index)

train_df = df.iloc[:train_index]
test_df = df.iloc[train_index:train_index + test_index]
val_df = df.iloc[train_index + test_index:]


(49620, 3)
42177 4962 2481
49620


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2-medium")

# device_map: "auto"를 사용하면 모델의 각 레이어가 가능한 한 GPU에 자동으로 할당된다.
model = AutoModelForCausalLM.from_pretrained("openai-community/gpt2-medium")

Loading weights: 100%|██████████| 292/292 [00:00<00:00, 1914.09it/s]


In [9]:
# 데이터셋 InstructionDataset 생성
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # inInstruction/input 부분을 ignore하기 위한 길이 저장리스트
        self.instruction_lengths = [] 

        # 텍스트 토큰화
        self.encoded_texts = []
        for i in range(data.shape[0]):
            instruction_input = format_input_not_contain_response(data.iloc[i])
            response_text = f"\n\n### Response:\n{data.iloc[i]['output']}"
            self.encoded_texts.append(tokenizer.encode(instruction_input + response_text))

            # instruction/input 부분의 길이를 저장
            instruction_length = len(tokenizer.encode(instruction_input))
            self.instruction_lengths.append(instruction_length)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # instruction/input 부분의 길이도 리턴한다.
        return self.instruction_lengths[idx], self.encoded_texts[idx]


In [10]:
# 데이터셋의 길이를 같은 길이로 패딩하는 콜레이트 함수를 정의한다.
# 입력에서 +1 값을 타켓으로 리턴한다.
# 타켓의 첫 번째 패딩을 제외하고 나머지는 ignore_index로 설정한다.
#   - ignore_index는 손실 계산에서 무시되는 인덱스이다.
def custom_collate_fn(batch, eos_token_id=50256, device="cuda", ignore_index=-100, allow_max_length=None):
    # 배치에서 최대 길이를 찾는다.
    batch_max_length = max(len(item)+1 for instruction_length, item in batch)

    # 패딩이 추가된 입력 리스트
    inputs_list = []
    outputs_list = []

    for instruction_length, item in batch:
        new_item = item.copy()
        new_item += [eos_token_id]
        # 최대 길이까지 패딩토큰을 추가한다.
        paded = (new_item + [eos_token_id] * (batch_max_length - len(new_item)))
        inputs = torch.tensor(paded[:-1])  # max_length - 1까지의 토큰을 입력으로 사용한다.
        outputs = torch.tensor(paded[1:])  # max_length까지 의 토큰을 타겟으로 사용한다.
        
        # 첫 번째 패딩 토큰을 제외하고 나머지는 ignore_index로 설정한다.
        mask = outputs == eos_token_id # outputs에서 패딩토큰의 위치를 찾는다.
        indices = torch.nonzero(mask).squeeze() # 패딩토큰의 위치를 인덱스로 변환한다.
        if indices.numel() > 1: # 패딩토큰이 존재하면
            outputs[indices[1:]] = ignore_index # 첫 번째 패딩토큰을 제외하고 나머지는 ignore_index로 설정한다.

        # 타깃에서 모든 입력 토큰을 ignore_index로 설정한다.
        outputs[:instruction_length-1] = ignore_index

        # 최대길이로 자르기
        if allow_max_length is not None:
            inputs = inputs[:allow_max_length]
            outputs = outputs[:allow_max_length]

        inputs_list.append(inputs)
        outputs_list.append(outputs)

    # 입력 리스트를 텐서 스텍으로 변환하고 타깃 장치로 전송한다.
    inputs_tensor = torch.stack(inputs_list).to(device)
    outputs_tensor = torch.stack(outputs_list).to(device)
    return inputs_tensor, outputs_tensor


In [40]:
# 데이터 로더 만들기
import torch
from torch.utils.data import DataLoader

num_workers = 0 # 데이터 로더에서 사용할 워커 수를 설정한다. 0이면 메인 프로세스에서 데이터를 로드한다.
batch_size = 1 # 배치 사이즈를 설정한다.

torch.manual_seed(123) # 랜덤 시드를 설정한다.

train_dataset = InstructionDataset(train_df, tokenizer)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=custom_collate_fn,
    num_workers=num_workers,
    drop_last=True
)

val_dataset = InstructionDataset(val_df, tokenizer)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=custom_collate_fn,
    num_workers=num_workers,
    drop_last=True
)

test_dataset = InstructionDataset(test_df, tokenizer)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=custom_collate_fn,
    num_workers=num_workers,
    drop_last=True
)


In [23]:
# 훈련데이터 로더
for batch in train_dataloader:
    inputs, targets = batch
    print(inputs.shape, targets.shape)

torch.Size([8, 958]) torch.Size([8, 958])
torch.Size([8, 292]) torch.Size([8, 292])
torch.Size([8, 564]) torch.Size([8, 564])
torch.Size([8, 434]) torch.Size([8, 434])
torch.Size([8, 585]) torch.Size([8, 585])
torch.Size([8, 446]) torch.Size([8, 446])
torch.Size([8, 442]) torch.Size([8, 442])
torch.Size([8, 430]) torch.Size([8, 430])
torch.Size([8, 385]) torch.Size([8, 385])
torch.Size([8, 456]) torch.Size([8, 456])
torch.Size([8, 426]) torch.Size([8, 426])
torch.Size([8, 845]) torch.Size([8, 845])
torch.Size([8, 333]) torch.Size([8, 333])
torch.Size([8, 531]) torch.Size([8, 531])
torch.Size([8, 280]) torch.Size([8, 280])
torch.Size([8, 496]) torch.Size([8, 496])
torch.Size([8, 347]) torch.Size([8, 347])
torch.Size([8, 515]) torch.Size([8, 515])
torch.Size([8, 557]) torch.Size([8, 557])
torch.Size([8, 381]) torch.Size([8, 381])
torch.Size([8, 358]) torch.Size([8, 358])
torch.Size([8, 479]) torch.Size([8, 479])
torch.Size([8, 631]) torch.Size([8, 631])
torch.Size([8, 507]) torch.Size([8

In [42]:
# 트랜스포머에서 제공하는 pipline를 사용하여 모델을 로드한다.
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,  # 전체 텍스트를 반환하지 않도록 설정
    max_new_tokens=100,      # 생성할 최대 토큰 수를 100으로 설정
    do_sample=False,          # 샘플링을 사용하여 텍스트를 생성하도록 설정
)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [21]:
# model val_data[0]을 사용하여 평가 실행
input_text = format_input(val_df.iloc[0])
print(input_text)


Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
수동태를 사용하지 않고 다음 문장을 다시 작성합니다:
회의는 내일로 예정되어 있었습니다.

### Response:
내일 회의가 예정되어 있습니다.


In [22]:
# model val_data[0]을 사용하여 평가 실행
input_text = format_input_not_contain_response(val_df.iloc[0])
print(input_text)


Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
수동태를 사용하지 않고 다음 문장을 다시 작성합니다:
회의는 내일로 예정되어 있었습니다.


In [23]:
generated_text = generator(input_text, max_new_tokens=100, num_return_sequences=1, truncation=True)[0]['generated_text']
print(generated_text)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.




아니의 사용하지 않고 일을 피적을 있었습니다.

아니의 사용하지 않고 다음 문장을 다시 작


In [38]:
# 손실을 측정할 함수를 만든다.
def cal_loss_loader(data_loader, model, device, num_batch=None):
    total_loss = 0.

    if num_batch is None:   
        num_batch = len(data_loader)
    else:
        num_batch = min(num_batch, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batch:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            logits = model(input_batch)
            loss = torch.nn.functional.cross_entropy(logits.logits.flatten(0, 1), target_batch.flatten())
            total_loss += loss.item()

        else:
            break

        #print(f"Batch {i+1}/{num_batch}: Loss = {total_loss / (i+1):.4f}")

    return total_loss / num_batch



In [25]:
# 훈력을 시작하기전  train_loader, val_loader의 손실을 예측한다.

model.to("cuda")

torch.manual_seed(123) # 랜덤 시드를 설정한다.

with torch.no_grad():
    train_loss = cal_loss_loader(train_dataloader, model, device="cuda", num_batch=10)
    val_loss = cal_loss_loader(val_dataloader, model, device="cuda", num_batch=10)

print(f"Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

Train Loss: 1.7066, Validation Loss: 1.6916


In [34]:
model.lm_head

Linear(in_features=1024, out_features=50257, bias=False)

In [35]:
import torch
# 모델의 출력을 이진 클래스(스팸 또는 스팸 아님)을 매핑하는 출력 레이어로 교체한다.
#   - (lm_head): Linear(in_features=1792, out_features=128256, bias=False)

# 모델을 동결한다.
for param in model.parameters():
    param.requires_grad = False

# 마지막 rotary_emb, norm을 훈련 가능하도록 설정한다.
for param in model.transformer.ln_f.parameters():
    param.requires_grad = True

for param in model.lm_head.parameters():
    param.requires_grad = True    


In [23]:
# 모델 평가 함수
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = cal_loss_loader(train_loader, model, device, num_batch=eval_iter)
        val_loss = cal_loss_loader(val_loader, model, device, num_batch=eval_iter)
    model.train()
    return train_loss, val_loss

In [37]:
evaluate_model(model, train_dataloader, val_dataloader, device="cuda", eval_iter=5)

(1.693503189086914, 1.65860595703125)

In [35]:
def calc_loss_batch(input_batch, target_batch, model, device):
    print(f"calc_loss_batch device: {device}")
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    print(111)
    logits = model(input_batch)
    print(222)
    loss = torch.nn.functional.cross_entropy(logits.logits.flatten(0, 1), target_batch.flatten())
    print(333)
    return loss

In [20]:
# 모델 학습 함수를 정의한다.
def train_simple_model(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter, input_text, tokenizer):

    # 손실 및 처리하한 샘플 수 리스트 초기화
    train_losses, val_losses = [], []
    example_seen, global_step = 0, -1

    # 메인 학습 루프
    for epoch in range(num_epochs):
        model.train() # 모델 훈련모드로 전환

        for i, (input_batch, target_batch) in enumerate(train_loader):
            optimizer.zero_grad() # 이전 배치에서 얻은 손실 재설정
            loss = calc_loss_batch(input_batch, target_batch, model, device) # 손실 계산.
            loss.backward() # 손실 그레이디언트 계산
            optimizer.step() # 손실 그레이드언트를 사용하여 모델 가중치 업데이트
            example_seen += input_batch.size(0) # 처리한 샘플 수 업데이트
            global_step += 1 # 전역 스텝 수 업데이트

            # 평가 주기마다 검증 손실 및 정확도 계산
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Epoch [{epoch+1}/{num_epochs}], Step [{global_step}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

      
    return train_losses, val_losses, example_seen

In [ ]:
import time

device = "cuda" if torch.cuda.is_available() else "cpu"

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.001)

num_epochs = 1


input_text = format_input_not_contain_response(val_df.iloc[0])

print("정답:\n", val_df.iloc[0]['output'])

train_losses, val_losses, tokens_seen = train_simple_model(
    model, train_dataloader, val_dataloader, optimizer, device,
    num_epochs=num_epochs, eval_freq=1, eval_iter=1,
    input_text=input_text, tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"훈련 소요 시간: {execution_time_minutes:.2f}분")

In [15]:
# 모델에서 평가할 알파카 형식 샘플 프롬프트를 생성한다.
sample_text = format_input(val_df.iloc[0])
print(sample_text)


Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
수동태를 사용하지 않고 다음 문장을 다시 작성합니다:
회의는 내일로 예정되어 있었습니다.

### Response:
내일 회의가 예정되어 있습니다.


In [16]:
sample_data = val_df.iloc[:1]
print(sample_data)

                                             instruction input  \
47139  수동태를 사용하지 않고 다음 문장을 다시 작성합니다:\n회의는 내일로 예정되어 있었...         

                  output  
47139  내일 회의가 예정되어 있습니다.  


In [31]:
# InstructionDataset으로 데이터 셋을 생성한다.
sample_dataset = InstructionDataset(sample_data, tokenizer)
print("샘플 데이터셋 길이:", len(sample_dataset)) 

#  sample_dataset의 첫 번째 샘플을 가져와서 입력과 타깃을 확인한다.
instruction_length, encoded_text = sample_dataset[0]
print("instruction_length:", instruction_length)
print("encoded_text:", encoded_text)

샘플 데이터셋 길이: 1
instruction_length: 127
encoded_text: [21106, 318, 281, 12064, 326, 8477, 257, 4876, 13, 16594, 257, 2882, 326, 20431, 32543, 262, 2581, 13, 198, 198, 21017, 46486, 25, 198, 168, 230, 246, 167, 237, 247, 169, 225, 250, 167, 98, 120, 23821, 8955, 168, 248, 102, 47991, 246, 168, 100, 222, 23821, 243, 232, 166, 111, 254, 31619, 233, 97, 35975, 234, 31619, 105, 116, 168, 252, 98, 35975, 226, 31619, 233, 97, 168, 233, 250, 23821, 252, 239, 168, 226, 109, 47991, 102, 46695, 230, 46695, 97, 25, 198, 169, 248, 234, 35975, 246, 167, 232, 242, 31619, 224, 112, 35975, 120, 167, 94, 250, 23821, 246, 230, 168, 254, 243, 167, 238, 246, 168, 244, 112, 23821, 252, 230, 168, 245, 230, 168, 232, 113, 46695, 230, 46695, 97, 13, 198, 198, 21017, 18261, 25, 198, 167, 224, 112, 35975, 120, 220, 169, 248, 234, 35975, 246, 166, 108, 222, 23821, 246, 230, 168, 254, 243, 167, 238, 246, 168, 244, 112, 23821, 252, 230, 168, 232, 113, 46695, 230, 46695, 97, 13]


In [1]:
import torch
import gc


# 2. 파이썬 가비지 컬렉션 실행
gc.collect()

# 3. 파이토치 CUDA 캐시 메모리 반환
torch.cuda.empty_cache()

In [36]:
from torch.utils.data import DataLoader
torch.manual_seed(123) # 랜덤 시드를 설정한다.

# sample_dataset의 첫 번째 샘플을 DataLoader를 사용하여 배치로 가져와서 입력과 타깃을 확인한다.
sample_loader = DataLoader(
    sample_dataset,
    batch_size=1,
    collate_fn=custom_collate_fn,
    num_workers=0
)
print("샘플 데이터 로더:", sample_loader)

for i, (input_batch, target_batch) in enumerate(sample_loader):
    print(f"Batch {i+1}:")
    #print("Input Batch:", input_batch)
    print("Target Batch:", target_batch)
    # Target Batch를 디코딩하여 출력한다.
    # ignore_index(-100) 제거 후 CPU 리스트로 변환해서 디코딩
    valid_target_ids = target_batch[0][target_batch[0] >= 0].detach().cuda().tolist()
    decoded_target = tokenizer.decode(valid_target_ids, skip_special_tokens=True)
    #print("Decoded Target:", decoded_target)
    break  # 첫 번째 배치만 확인

샘플 데이터 로더: <torch.utils.data.dataloader.DataLoader object at 0x78f639bded20>
Batch 1:
Target Batch: tensor([[ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  

In [39]:
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

start_time = time.time()

torch.manual_seed(123)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.001)

num_epochs = 100


input_text = format_input_not_contain_response(val_df.iloc[0])

print("정답:\n", val_df.iloc[0]['output'])

train_losses, val_losses, tokens_seen = train_simple_model(
    model, sample_loader, sample_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    input_text=input_text, tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"훈련 소요 시간: {execution_time_minutes:.2f}분")

Using device: cuda
정답:
 내일 회의가 예정되어 있습니다.
calc_loss_batch device: cuda
111
222
333
Epoch [1/100], Step [0], Train Loss: 0.1782, Val Loss: 0.1782
calc_loss_batch device: cuda
111
222
333
calc_loss_batch device: cuda
111
222
333
calc_loss_batch device: cuda
111
222
333
calc_loss_batch device: cuda
111
222
333
calc_loss_batch device: cuda
111
222
333
Epoch [6/100], Step [5], Train Loss: 0.0576, Val Loss: 0.0576
calc_loss_batch device: cuda
111
222
333
calc_loss_batch device: cuda
111
222
333
calc_loss_batch device: cuda
111
222
333
calc_loss_batch device: cuda
111
222
333
calc_loss_batch device: cuda
111
222
333
Epoch [11/100], Step [10], Train Loss: 0.0108, Val Loss: 0.0108
calc_loss_batch device: cuda
111
222
333
calc_loss_batch device: cuda
111
222
333
calc_loss_batch device: cuda
111
222
333
calc_loss_batch device: cuda
111
222
333
calc_loss_batch device: cuda
111
222
333
Epoch [16/100], Step [15], Train Loss: 0.0414, Val Loss: 0.0414
calc_loss_batch device: cuda
111
222
333
calc_loss

In [ ]:
model

In [40]:
# model val_data[0]을 사용하여 평가 실행
input_text = format_input_not_contain_response(val_df.iloc[0])
print(input_text)

Below is an instruction that describes a task.Write a response that appropriately completes the request.

### Instruction:
수동태를 사용하지 않고 다음 문장을 다시 작성합니다:
회의는 내일로 예정되어 있었습니다.


In [43]:
generated_text = generator(input_text, max_new_tokens=1000, num_return_sequences=1)[0]['generated_text']
print(generated_text)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=1000) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.




### Response:
내일 회의가 예정되어 있습니다.


In [44]:
sample_data = val_df.iloc[:1]
print(sample_data)

                                             instruction input  \
47139  수동태를 사용하지 않고 다음 문장을 다시 작성합니다:\n회의는 내일로 예정되어 있었...         

                  output  
47139  내일 회의가 예정되어 있습니다.  
